# SPY daily and five-minute chart experiments
Compare the 30-minute SPY direction forecast from the five-minute chart alone, both chart images, and both images with six price distances in common units. Select checkpoints on 2024 after fitting on 2020–2023. Compare the locked models on 2025. The 2026 holdout stays sealed. Each epoch records fit loss and accuracy, development accuracy, balanced accuracy, AUC and log loss. A T4 GPU is selected in the notebook metadata.

In [ ]:
# @title Load the prepared data
from pathlib import Path
import hashlib,zipfile,sys,subprocess,requests,shutil
from google.colab import files
root=Path('/content/paired_overnight')
root.mkdir(parents=True,exist_ok=True)
data=root/'dataset'
data.mkdir(exist_ok=True)
archive=Path('/content/SPY_Paired_2020_2025_Data.zip')
if not archive.exists():
    upload=files.upload()
    selected=next(Path(n) for n in upload if n.endswith('.zip'))
    if selected.resolve()!=archive.resolve():shutil.copyfile(selected,archive)
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='f5d5493c34d4d55dc734cfd6349bc9e72680caee906a1083442116b54ad18209'
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    z.extractall(data)
for name in ['train_paired_charts.py','paired_decision_charts.py']:
    resp=requests.get('https://raw.githubusercontent.com/jamelof23/SPY-EMA-V2/66585090699610f1942b02d3cda89ad851ae1354/src/'+name,timeout=60)
    resp.raise_for_status()
    (root/name).write_bytes(resp.content)
import torch
print('Dataset verified.',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# @title Train the three models and save each epoch
result=root/'results'
with (root/'training.log').open('w') as log:
    command=[sys.executable,'-u',str(root/'train_paired_charts.py'),'--data',str(data),'--out',str(result),'--epochs','4']
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        if process.wait(): raise RuntimeError('Training stopped. Inspect the log. The most recent completed epoch is saved.')

In [ ]:
# @title Save logs, 2025 predictions and all model checkpoints
import base64, json, math
from IPython.display import display
pack=root/'SPY_Paired_CNN_Checkpoints.zip'
with zipfile.ZipFile(pack,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2) as bundle:
    for item in sorted(result.rglob('*')):
        if item.is_file():bundle.write(item,item.relative_to(result))
    bundle.write(root/'training.log','training.log')
    bundle.write(root/'train_paired_charts.py','train_paired_charts.py')
buf=pack.read_bytes()
with zipfile.ZipFile(pack) as z:assert z.testzip() is None
step=375000
digest=hashlib.sha256(buf).hexdigest()
for i in range(math.ceil(len(buf)/step)):
    payload=dict(format='SPY-paired-CNN-checkpoints-v1',part=i,count=math.ceil(len(buf)/step),sha256=digest,payload=base64.b64encode(buf[i*step:(i+1)*step]).decode())
    display({'application/json':payload,'text/plain':f'weights block {i+1}'},raw=True)
print('Saved checkpoint archive',pack,pack.stat().st_size,digest)
print((result/'summary.json').read_text())